# Etapa 2 — Modelado con Validación Segura
## Predicción de Falla Cardíaca

**Objetivo de este notebook:**
1. Partir de la conclusión de la Etapa 1 (el mejor modelo encontrado en el ranking).
2. Reconstruir el flujo completo con partición train/test **antes** de cualquier escalado.
3. Ajustar el modelo elegido dentro de un `Pipeline` + `GridSearchCV`, con una grilla más fina alrededor de sus mejores hiperparámetros.
4. Evaluar de forma rigurosa: matriz de confusión, curva ROC, AUC, y validación cruzada.
5. Dejar el `best_estimator_` listo para exportarlo en la Etapa 3.


In [1]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go

from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    roc_auc_score, accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, roc_curve, classification_report
)

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

## 1. Carga de datos

Mismo dataset que en la Etapa 1. Ajusta la ruta si es necesario.

In [6]:
df = pd.read_csv(
    r"C:\Users\Katherin Barrera\Downloads\heart-disease-mlops\heart-disease-mlops\notebooks\heart.csv"
)
target_col = "HeartDisease"  # ajusta el nombre si tu dataset usa otro

X = df.drop(columns=[target_col])
y = df[target_col]

numeric_features = X.select_dtypes(include=["int64", "float64"]).columns.tolist()
categorical_features = X.select_dtypes(include=["object"]).columns.tolist()

print("Numéricas:", numeric_features)
print("Categóricas:", categorical_features)
print("Shape:", X.shape)

Numéricas: ['Age', 'RestingBP', 'Cholesterol', 'FastingBS', 'MaxHR', 'Oldpeak']
Categóricas: ['Sex', 'ChestPainType', 'RestingECG', 'ExerciseAngina', 'ST_Slope']
Shape: (918, 11)


## 2. Partición train/test (ANTES de cualquier escalado)

Esta es la regla de oro que validamos en la Etapa 1: dividimos primero, y todo el preprocesamiento se ajusta únicamente con los datos de entrenamiento.

In [7]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

print("Train:", X_train.shape, " Test:", X_test.shape)
print("Proporción de clases en train:")
print(y_train.value_counts(normalize=True))
print("Proporción de clases en test:")
print(y_test.value_counts(normalize=True))

Train: (734, 11)  Test: (184, 11)
Proporción de clases en train:
HeartDisease
1    0.553134
0    0.446866
Name: proportion, dtype: float64
Proporción de clases en test:
HeartDisease
1    0.554348
0    0.445652
Name: proportion, dtype: float64


## 3. Preprocesador

Mismo `ColumnTransformer` que en la Etapa 1: escalado para numéricas, one-hot para categóricas.

In [8]:
preprocessor = ColumnTransformer(transformers=[
    ("num", StandardScaler(), numeric_features),
    ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features)
])

## 4. Selección del modelo final

> **Reemplaza el modelo y la grilla de abajo con el que ganó el ranking de la Etapa 1.**
Según el ranking de la Etapa 1, el modelo ganador fue **KNeighborsClassifier** (AUC=0.953), seguido de cerca por SVC (0.949).

In [9]:
from sklearn.neighbors import KNeighborsClassifier

# Ganador del ranking de la Etapa 1: KNeighborsClassifier (AUC=0.953)
final_model = KNeighborsClassifier()

# Grilla más fina, centrada alrededor de los mejores hiperparámetros de la Etapa 1
param_grid = {
    "model__n_neighbors": [3, 5, 7, 9, 11, 13],
    "model__weights": ["uniform", "distance"],
    "model__p": [1, 2],  # 1 = distancia Manhattan, 2 = Euclidiana
}

pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", final_model)
])

## 5. GridSearchCV con validación cruzada estratificada

Usamos `StratifiedKFold` para que cada fold mantenga la misma proporción de clases que el dataset completo — importante porque la variable objetivo puede estar algo desbalanceada.

In [10]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

grid = GridSearchCV(
    pipeline,
    param_grid=param_grid,
    cv=cv,
    scoring="roc_auc",
    n_jobs=-1,
    refit=True
)

grid.fit(X_train, y_train)

print("Mejores hiperparámetros:", grid.best_params_)
print("Mejor AUC en CV:", grid.best_score_)

Mejores hiperparámetros: {'model__n_neighbors': 13, 'model__p': 1, 'model__weights': 'distance'}
Mejor AUC en CV: 0.9244582086587506


In [11]:
best_model = grid.best_estimator_

## 6. Evaluación en el conjunto de test

In [12]:
y_pred = best_model.predict(X_test)
y_proba = best_model.predict_proba(X_test)[:, 1]

metrics = {
    "Accuracy": accuracy_score(y_test, y_pred),
    "Precision": precision_score(y_test, y_pred),
    "Recall": recall_score(y_test, y_pred),
    "F1-score": f1_score(y_test, y_pred),
    "AUC": roc_auc_score(y_test, y_proba),
}

for name, value in metrics.items():
    print(f"{name}: {value:.4f}")

Accuracy: 0.9185
Precision: 0.9307
Recall: 0.9216
F1-score: 0.9261
AUC: 0.9433


In [13]:
print(classification_report(y_test, y_pred, target_names=["No enfermedad", "Enfermedad"]))

               precision    recall  f1-score   support

No enfermedad       0.90      0.91      0.91        82
   Enfermedad       0.93      0.92      0.93       102

     accuracy                           0.92       184
    macro avg       0.92      0.92      0.92       184
 weighted avg       0.92      0.92      0.92       184



## 7. Matriz de confusión 

In [14]:
cm = confusion_matrix(y_test, y_pred)
labels = ["No enfermedad", "Enfermedad"]

fig = go.Figure(data=go.Heatmap(
    z=cm,
    x=labels,
    y=labels,
    colorscale="Blues",
    showscale=True,
    hovertemplate="Real: %{y}<br>Predicho: %{x}<br>Conteo: %{z}<extra></extra>"
))

annotations = []
for i, row_label in enumerate(labels):
    for j, col_label in enumerate(labels):
        annotations.append(dict(
            x=col_label, y=row_label,
            text=str(cm[i, j]),
            showarrow=False,
            font=dict(color="white" if cm[i, j] > cm.max() / 2 else "black", size=16)
        ))

fig.update_layout(
    title=f"Matriz de confusión — {best_model.named_steps['model'].__class__.__name__}",
    xaxis_title="Predicción",
    yaxis_title="Valor real",
    annotations=annotations,
    width=500,
    height=500
)
fig.update_yaxes(autorange="reversed")
fig.show()

## 8. Curva ROC 

In [15]:
fpr, tpr, thresholds = roc_curve(y_test, y_proba)
auc_value = roc_auc_score(y_test, y_proba)

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=fpr, y=tpr, mode="lines",
    name=f"{best_model.named_steps['model'].__class__.__name__} (AUC={auc_value:.3f})",
    hovertemplate="FPR: %{x:.3f}<br>TPR: %{y:.3f}<extra></extra>"
))
fig.add_trace(go.Scatter(
    x=[0, 1], y=[0, 1], mode="lines",
    name="Azar", line=dict(dash="dash", color="gray")
))

fig.update_layout(
    title="Curva ROC — Modelo final",
    xaxis_title="False Positive Rate",
    yaxis_title="True Positive Rate",
    width=650,
    height=550
)
fig.show()

## 9. Estabilidad del modelo: validación cruzada final

Revisamos qué tan estable es el AUC del mejor modelo a través de los folds, para descartar que el buen resultado se deba a una partición particularmente favorable.

In [16]:
cv_scores = cross_val_score(best_model, X_train, y_train, cv=cv, scoring="roc_auc", n_jobs=-1)

print("AUC por fold:", np.round(cv_scores, 4))
print(f"Media: {cv_scores.mean():.4f}  |  Desviación estándar: {cv_scores.std():.4f}")

fig = px.box(y=cv_scores, points="all", title="Distribución de AUC en validación cruzada (5 folds)")
fig.update_layout(yaxis_title="AUC", xaxis_title="", width=450, height=500)
fig.show()

AUC por fold: [0.9302 0.8602 0.9452 0.956  0.9307]
Media: 0.9245  |  Desviación estándar: 0.0336


## 10. Importancia de features (si el modelo lo soporta)

Útil para entender qué variables pesan más en la predicción — relevante también para justificar el modelo en el informe.

In [17]:
model_step = best_model.named_steps["model"]

if hasattr(model_step, "feature_importances_"):
    feature_names = best_model.named_steps["preprocessor"].get_feature_names_out()
    importances = model_step.feature_importances_

    importance_df = pd.DataFrame({
        "Feature": feature_names,
        "Importancia": importances
    }).sort_values("Importancia", ascending=True)

    fig = px.bar(
        importance_df, x="Importancia", y="Feature", orientation="h",
        title="Importancia de features", color="Importancia", color_continuous_scale="viridis"
    )
    fig.update_layout(coloraxis_showscale=False, height=max(400, 20 * len(feature_names)))
    fig.show()
else:
    print(f"{model_step.__class__.__name__} no expone feature_importances_ directamente "
          f"(por ejemplo LogisticRegression usa coeficientes, SVC/KNN no tienen esta métrica).")

KNeighborsClassifier no expone feature_importances_ directamente (por ejemplo LogisticRegression usa coeficientes, SVC/KNN no tienen esta métrica).


## Conclusiones

- El modelo final fue entrenado respetando la partición train/test previa al escalado, evitando data leakage.
- Las métricas en test y la validación cruzada confirman que el desempeño es consistente (baja varianza entre folds).
- El `best_estimator_` (`best_model`) queda listo para exportarse con `joblib.dump(best_model, "model.joblib")` en la **Etapa 3**, junto con la construcción de la API en FastAPI.


## 11. Exportación del modelo para despliegue

Guardamos el `Pipeline` completo (preprocesador + modelo KNN) con `joblib`, listo para que la API de la Etapa 3 lo cargue y haga predicciones sobre datos nuevos sin re-entrenar nada.

In [18]:
import joblib

# Exportamos el pipeline completo (preprocesador + modelo) para usarlo en la API
joblib.dump(best_model, "../model.joblib")
print("Modelo exportado a model.joblib")

# Guardamos también la lista de columnas esperadas, para validarlas en la API
joblib.dump(list(X.columns), "../feature_columns.joblib")
print("Columnas exportadas:", list(X.columns))


Modelo exportado a model.joblib
Columnas exportadas: ['Age', 'Sex', 'ChestPainType', 'RestingBP', 'Cholesterol', 'FastingBS', 'RestingECG', 'MaxHR', 'ExerciseAngina', 'Oldpeak', 'ST_Slope']
